# SIS-API-01 — Entregables del 18/11/2026
## Contratos y endpoints
Notebook complementario. **No reemplaza el piloto original.** Datos sintéticos o ilustrativos; no constituyen validación real. Ejecutar en Colab/Jupyter y descargar la carpeta de salida.


In [ ]:
import json
from pathlib import Path
out=Path("salida_sis_api");out.mkdir(exist_ok=True)
# Contrato mínimo compartido; escala H,E,V en [0,1].
ejemplo={"id_zona":"PER-01","periodo":"2026-01","riesgo":"vial","H":0.6,"E":0.5,"V":0.4,"origen":"sintetico","version":"0.1"}
assert all(0<=ejemplo[k]<=1 for k in "HEV")
(out/"zona_hev_ejemplo.json").write_text(json.dumps(ejemplo,indent=2),encoding="utf-8")
codigo='''from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal
app=FastAPI(title="RISKTWIN contrato demo")
class ZonaHEV(BaseModel):
 id_zona:str
 periodo:str
 riesgo:Literal["vial","inundacion","seguridad"]
 H:float=Field(ge=0,le=1)
 E:float=Field(ge=0,le=1)
 V:float=Field(ge=0,le=1)
 origen:str
 version:str
@app.get("/health")
def health():return {"estado":"ok"}
@app.post("/zonas/validar")
def validar(z:ZonaHEV):return {"valido":True,"id_zona":z.id_zona}
@app.post("/indicadores/calcular")
def calcular(z:ZonaHEV):return {"id_zona":z.id_zona,"indice_demo":round(.4*z.H+.3*z.E+.3*z.V,4)}
'''
(out/"app_demo.py").write_text(codigo,encoding="utf-8")
print("Tres endpoints definidos: GET /health; POST /zonas/validar; POST /indicadores/calcular")
try:
 from fastapi.testclient import TestClient
 ns={};exec(codigo,ns)
 c=TestClient(ns["app"])
 assert c.get("/health").status_code==200
 assert c.post("/zonas/validar",json=ejemplo).status_code==200
 assert c.post("/indicadores/calcular",json=ejemplo).status_code==200
 print("Prueba funcional: OK")
except ImportError:
 print("Instalar fastapi y httpx para ejecutar prueba: pip install fastapi httpx")